# Zebrafish SRA (NCBI) pull + test download

This notebook runs the heavy steps on `sequoia` via `ssh`, using the shared repo clone at `/home/zebrafish` (same code as your local clone).
Outputs go to `zebrafish/metadata/` (tracked) and `zebrafish/data/` + `zebrafish/tools/` (gitignored).


In [56]:
# Quick sanity check: connect to sequoia and print host/user/pwd.
!ssh -X pzg8794@sequoia.rit.edu "hostname; whoami; pwd"

RIT information technology resources are for the use of the RIT community only. 
By using RIT information technology resources you acknowledge that you have read 
and comply with RIT's Code of Conduct for Computer and Network Use and RIT's 
Information Security Policy and Standards. Use of RIT information technology 
resources may be monitored and unauthorized use is strictly prohibited.
sequoia
pzg8794
/home/pzg8794


## Local vs server repo (same codebase)

This prints the git branch + commit hash locally and on `sequoia:/home/zebrafish` so you can confirm both are in sync.


In [57]:
%%bash
set -euo pipefail
# Compare local vs server repo commit so you know both clones match.
# If they differ, run the server setup/pull step before continuing.


echo "LOCAL" 
echo "-----"
# Notebook is in zebrafish/; this finds the repo root reliably.
GIT_ROOT="$(git rev-parse --show-toplevel)"
echo "git_root: $GIT_ROOT"
cd "$GIT_ROOT"
git status -sb || true
git rev-parse HEAD

echo
echo "SERVER (sequoia:/home/zebrafish)"
echo "----------------------------"
SSH="ssh -X pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

$SSH "REMOTE_REPO=$REMOTE_REPO bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"
git status -sb || true
git rev-parse HEAD
EOF

LOCAL
-----
git_root: /Users/pitergarcia/DataScience/Semester5/BIOL550/group_project
## main...origin/main
 M zebrafish/ZEBRAFISH_FASTQ_DOWNLOAD_STEP_BY_STEP.md
 M zebrafish/zebrafish_github_setup_and_script_walkthrough.ipynb
 M zebrafish/zebrafish_sra_api_test_download.ipynb
0fc70fa517ef5e682cc60fd3dff38250fbee4afe

SERVER (sequoia:/home/zebrafish)
----------------------------


RIT information technology resources are for the use of the RIT community only. 
By using RIT information technology resources you acknowledge that you have read 
and comply with RIT's Code of Conduct for Computer and Network Use and RIT's 
Information Security Policy and Standards. Use of RIT information technology 
resources may be monitored and unauthorized use is strictly prohibited.


## main...origin/main
0fc70fa517ef5e682cc60fd3dff38250fbee4afe


## 0) Ensure the shared server directory is the repo

Make sure `/home/zebrafish` on `sequoia` is a git clone of the repo (and up to date) so all server runs use the same scripts.


In [58]:
%%bash
set -euo pipefail

# Bootstrap /home/zebrafish as the BIOL550-group_project repo clone (server-side).
# Also keeps it up-to-date (git pull) so new notebook/scripts are available.

SSH="ssh -X pzg8794@sequoia.rit.edu"
REPO_DIR="/home/zebrafish"
REPO_URL_HTTPS="https://github.com/pzg8794/BIOL550-group_project.git"

$SSH "REPO_DIR=$REPO_DIR REPO_URL_HTTPS=$REPO_URL_HTTPS bash -s" <<'EOF'
set -euo pipefail
cd "$REPO_DIR"

if [ -d .git ]; then
  echo "OK: $REPO_DIR is already a git repo"

  # If git refuses due to 'dubious ownership', print the exact fix and stop.
  if ! git status -sb >/dev/null 2>&1; then
    echo "git status failed. If you see 'detected dubious ownership', run:" 
    echo "  git config --global --add safe.directory $REPO_DIR"
    git status -sb || true
    exit 2
  fi

  echo
  echo "Updating repo (fast-forward only)..."
  git fetch origin
  if ! git pull --ff-only; then
    echo "WARN: could not fast-forward. Repo may have local changes or diverged." 
    git status -sb || true
    echo "Resolve manually, then re-run this cell."
    exit 2
  fi

  echo
  git status -sb || true
  echo "server HEAD: $(git rev-parse HEAD)"
  exit 0
fi

echo "Bootstrapping repo in $REPO_DIR"
ts=$(date +%F_%H%M%S)
backup="$HOME/zebrafish_preclone_backups/$ts"
mkdir -p "$backup"

# If the directory has anything in it, move it to the backup folder first.
shopt -s dotglob nullglob
items=(*)
if [ ${#items[@]} -gt 0 ]; then
  echo "Backing up existing content to: $backup"
  mv -- "${items[@]}" "$backup/"
fi

git clone "$REPO_URL_HTTPS" .
echo "Cloned repo into $REPO_DIR"

git status -sb || true
echo "server HEAD: $(git rev-parse HEAD)"
EOF

RIT information technology resources are for the use of the RIT community only. 
By using RIT information technology resources you acknowledge that you have read 
and comply with RIT's Code of Conduct for Computer and Network Use and RIT's 
Information Security Policy and Standards. Use of RIT information technology 
resources may be monitored and unauthorized use is strictly prohibited.


OK: /home/zebrafish is already a git repo

Updating repo (fast-forward only)...
Already up to date.

## main...origin/main
server HEAD: 0fc70fa517ef5e682cc60fd3dff38250fbee4afe


## Outline

We (1) verify local vs server repo, (2) fetch RunInfo + pick a small run list, (3) download the run file(s), then (4) use SRA Toolkit to extract a small FASTQ subset.
Edit `ACC`, `N_TEST_RUNS`, and `MAX_SPOTS` in the code cells when switching datasets or test sizes.


In [59]:
%%bash
set -euo pipefail

# Server setup: run inside the repo root on sequoia (/home/zebrafish).
# Zebrafish workflow lives under the repo subdir: /home/zebrafish/zebrafish/

SSH="ssh -X pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

ACC="PRJNA1277581"
MAX_SPOTS="10000"

$SSH "hostname; whoami; pwd"

$SSH "REMOTE_REPO=$REMOTE_REPO ACC=$ACC MAX_SPOTS=$MAX_SPOTS bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"
export PATH="$HOME/bin:$PATH"
echo "fastq-dump:" $(command -v fastq-dump 2>/dev/null || echo missing)

echo "Repo dir: $(pwd)"
ls -la

test -d zebrafish || { echo 'ERROR: missing zebrafish/ directory in repo root'; exit 2; }
test -f zebrafish/scripts/get_zebrafish_data_sra.py || { echo 'ERROR: missing zebrafish/scripts/get_zebrafish_data_sra.py'; exit 2; }

echo "ACC=$ACC"
echo "MAX_SPOTS=$MAX_SPOTS"

META_DIR="zebrafish/metadata/$ACC"
DATA_TEST_DIR="zebrafish/data/$ACC"
mkdir -p "$META_DIR" "$DATA_TEST_DIR"

echo "META_DIR: $META_DIR"
echo "DATA_TEST_DIR: $DATA_TEST_DIR"
EOF

RIT information technology resources are for the use of the RIT community only. 
By using RIT information technology resources you acknowledge that you have read 
and comply with RIT's Code of Conduct for Computer and Network Use and RIT's 
Information Security Policy and Standards. Use of RIT information technology 
resources may be monitored and unauthorized use is strictly prohibited.


sequoia
pzg8794
/home/pzg8794


## What the server-setup cell does

It runs commands on `sequoia` in `/home/zebrafish` and ensures required paths/tools exist before downloading or extracting anything.
If anything is missing, fix it in the setup steps first (so later steps don’t fail halfway).


## 1) Check the workspace structure

This confirms we’re writing metadata and data into the expected subfolders.

In [60]:
%%bash
set -euo pipefail

# Server-side workspace check (repo root + zebrafish subdir).

SSH="ssh -X pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

ACC="PRJNA1277581"
MAX_SPOTS="10000"

$SSH "REMOTE_REPO=$REMOTE_REPO ACC=$ACC MAX_SPOTS=$MAX_SPOTS bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

PYTHON="/usr/bin/python3"
if [ ! -x "$PYTHON" ]; then
  PYTHON="$(command -v python3 || command -v python || true)"
fi

if [ -z "$PYTHON" ]; then
  echo 'ERROR: python3 not found on sequoia'
  exit 127
fi

echo "Using PYTHON=$PYTHON"
"$PYTHON" --version
"$PYTHON" - <<'PYCHK'
import sys
print('Python OK:', sys.version.split()[0])
PYCHK

"$PYTHON" - <<'PY'
from pathlib import Path
import os

repo = Path(os.environ['REMOTE_REPO'])
zroot = repo / 'zebrafish'
acc = os.environ['ACC']
max_spots = os.environ['MAX_SPOTS']

meta_dir = zroot / 'metadata' / acc
data_test_dir = zroot / 'data' / 'fastq' / 'subsets' / acc / f'spots_{max_spots}'

print('REPO_ROOT:', repo)
print('ZEBRAFISH_ROOT:', zroot)
print('META_DIR:', meta_dir)
print('DATA_TEST_DIR:', data_test_dir)

meta_dir.mkdir(parents=True, exist_ok=True)
data_test_dir.mkdir(parents=True, exist_ok=True)

for rel in ['scripts', 'metadata', 'notes', 'data']:
    p = zroot / rel
    print(rel, 'exists=', p.exists())
PY
EOF

RIT information technology resources are for the use of the RIT community only. 
By using RIT information technology resources you acknowledge that you have read 
and comply with RIT's Code of Conduct for Computer and Network Use and RIT's 
Information Security Policy and Standards. Use of RIT information technology 
resources may be monitored and unauthorized use is strictly prohibited.


Using PYTHON=/usr/bin/python3
Python 3.13.5
Python OK: 3.13.5
REPO_ROOT: /home/zebrafish
ZEBRAFISH_ROOT: /home/zebrafish/zebrafish
META_DIR: /home/zebrafish/zebrafish/metadata/PRJNA1277581
DATA_TEST_DIR: /home/zebrafish/zebrafish/data/fastq/subsets/PRJNA1277581/spots_10000
scripts exists= True
metadata exists= True
notes exists= True
data exists= True


**Step 1 (Server setup / workspace):** Connects to `sequoia`, switches into the repo at `/home/zebrafish`, and creates the expected zebrafish working folders under `zebrafish/metadata/<ACC>/` and `zebrafish/data/<ACC>/spots_<MAX_SPOTS>/`.

## 2) Fetch RunInfo (SRA) via API

We call the existing script, but skip re-fetching if `runinfo.csv` already exists (unless `FORCE_REFETCH_RUNINFO=True`).

In [61]:
%%bash
set -euo pipefail

# Fetch RunInfo on sequoia (server) using our script.

SSH="ssh -X pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

ACC="PRJNA1277581"
ORGANISM="Danio rerio"
FORCE_REFETCH_RUNINFO=0   # set to 1 to re-fetch

$SSH "REMOTE_REPO=$REMOTE_REPO ACC=$ACC FORCE_REFETCH_RUNINFO=$FORCE_REFETCH_RUNINFO bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

ORGANISM="Danio rerio"

PYTHON="/usr/bin/python3"
if [ ! -x "$PYTHON" ]; then
  PYTHON="$(command -v python3 || command -v python || true)"
fi

if [ -z "$PYTHON" ]; then
  echo 'ERROR: python3 not found on sequoia'
  exit 127
fi

echo "Using PYTHON=$PYTHON"
"$PYTHON" --version
"$PYTHON" - <<'PYCHK'
import sys
print('Python OK:', sys.version.split()[0])
PYCHK

OUT_DIR="zebrafish/metadata/$ACC"
RUNINFO="$OUT_DIR/runinfo.csv"
mkdir -p "$OUT_DIR"

if [ "$FORCE_REFETCH_RUNINFO" = "1" ] && [ -f "$RUNINFO" ]; then
  rm -f "$RUNINFO"
fi

if [ ! -f "$RUNINFO" ]; then
  echo 'Fetching RunInfo...'
  "$PYTHON" zebrafish/scripts/get_zebrafish_data_sra.py     --acc "$ACC"     --out-dir "$OUT_DIR"     --organism "$ORGANISM"     --library-strategy ""     --library-layout ""     --min-spots 0     --min-avg-length 0     --write-download-urls
else
  echo "RunInfo already exists, skipping: $RUNINFO"
fi

"$PYTHON" - <<'PY'
import csv
import os
from pathlib import Path
acc = os.environ['ACC']
runinfo = Path('zebrafish') / 'metadata' / acc / 'runinfo.csv'
rows = list(csv.DictReader(runinfo.read_text(encoding='utf-8').splitlines()))
print('runinfo rows:', len(rows))
print('first Run:', rows[0].get('Run') if rows else None)
PY
EOF

RIT information technology resources are for the use of the RIT community only. 
By using RIT information technology resources you acknowledge that you have read 
and comply with RIT's Code of Conduct for Computer and Network Use and RIT's 
Information Security Policy and Standards. Use of RIT information technology 
resources may be monitored and unauthorized use is strictly prohibited.


Using PYTHON=/usr/bin/python3
Python 3.13.5
Python OK: 3.13.5
RunInfo already exists, skipping: zebrafish/metadata/PRJNA1277581/runinfo.csv
runinfo rows: 30
first Run: SRR34002439


**Step 2 (Fetch RunInfo):** Uses the server-side script to fetch SRA RunInfo for `ACC` into `zebrafish/metadata/<ACC>/runinfo.csv` (skips if it already exists) and prints a quick sanity check (row count + first SRR).

## 3) Create the 5-run test list

We pick the 5 smallest runs (by `size_MB`) from RunInfo, excluding any SRRs in `EXCLUDE_RUNS`.

In [62]:
%%bash
set -euo pipefail

# Pick N smallest paired-end SRRs (by size_MB) from runinfo.csv and write runs.test5.smallest_sizeMB.txt.

SSH="ssh -X pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"
ACC="PRJNA1277581"
N_TEST_RUNS="1"   # change to 5 when ready

$SSH "REMOTE_REPO=$REMOTE_REPO ACC=$ACC N_TEST_RUNS=$N_TEST_RUNS bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

PYTHON="/usr/bin/python3"
if [ ! -x "$PYTHON" ]; then
  PYTHON="$(command -v python3 || command -v python || true)"
fi

if [ -z "$PYTHON" ]; then
  echo 'ERROR: python3 not found on sequoia'
  exit 127
fi

echo "Using PYTHON=$PYTHON"
"$PYTHON" --version
"$PYTHON" - <<'PYCHK'
import sys
print('Python OK:', sys.version.split()[0])
PYCHK

"$PYTHON" - <<'PY'
import csv
import math
import os
from pathlib import Path

acc = os.environ['ACC']
n = int(os.environ['N_TEST_RUNS'])
exclude = {'SRR34002423', 'SRR34002425'}

runinfo = Path('zebrafish') / 'metadata' / acc / 'runinfo.csv'
out_file = Path('zebrafish') / 'metadata' / acc / 'runs.test5.smallest_sizeMB.txt'

if not runinfo.exists():
    raise FileNotFoundError(runinfo)

rows = list(csv.DictReader(runinfo.read_text(encoding='utf-8').splitlines()))
vals = []
for r in rows:
    if (r.get('LibraryLayout') or '').strip() != 'PAIRED':
        continue
    srr = (r.get('Run') or '').strip()
    if not srr or srr in exclude:
        continue
    try:
        size = float(r.get('size_MB') or 'nan')
    except Exception:
        size = float('nan')
    if math.isfinite(size):
        vals.append((size, srr))

vals.sort()
selected = [srr for _, srr in vals[:n]]
if len(selected) != n:
    raise SystemExit(f'Expected {n} paired-end runs, got {len(selected)}: {selected}')

out_file.write_text("\n".join(selected) + "\n", encoding="utf-8")
print('Wrote:', out_file)
print('Selected SRRs (smallest size_MB):')
print("\n".join(selected))
PY
EOF

RIT information technology resources are for the use of the RIT community only. 
By using RIT information technology resources you acknowledge that you have read 
and comply with RIT's Code of Conduct for Computer and Network Use and RIT's 
Information Security Policy and Standards. Use of RIT information technology 
resources may be monitored and unauthorized use is strictly prohibited.


Using PYTHON=/usr/bin/python3
Python 3.13.5
Python OK: 3.13.5
Wrote: zebrafish/metadata/PRJNA1277581/runs.test5.smallest_sizeMB.txt
Selected SRRs (smallest size_MB):
SRR34002427


## 4) Download run files via NCBI (from `runinfo.csv` -> `download_path`)

The ENA Portal API is not returning FASTQ links for these SRR accessions (it returns only the TSV header), so we download the run files using the NCBI-provided `download_path` in `zebrafish/metadata/<ACC>/runinfo.csv`.

This step downloads the run files into `zebrafish/data/<ACC>/spots_<MAX_SPOTS>/<SRR>/sra/`. The `MAX_SPOTS` value is kept for directory naming consistency (the run file is still the full run; the FASTQ subset extraction happens in Step 4b).


## Pre-flight: confirm download tools + metadata on sequoia

This checks that `curl` and `python3` exist on the server and that `runinfo.csv` (with `download_path`) is present for this BioProject.


In [63]:
%%bash
set -euo pipefail
# Pre-flight: verify curl/python3 and confirm runinfo.csv contains download_path URLs.
# If this fails, fix metadata/tools before downloading anything.


SSH="ssh -X pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

ACC="PRJNA1277581"

$SSH "REMOTE_REPO=$REMOTE_REPO ACC=$ACC bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

command -v curl >/dev/null 2>&1 || { echo "ERROR: missing curl"; exit 127; }
command -v python3 >/dev/null 2>&1 || { echo "ERROR: missing python3"; exit 127; }

echo "curl:    $(command -v curl)"
echo "python3: $(command -v python3)"

RUNINFO="zebrafish/metadata/$ACC/runinfo.csv"
if [ ! -f "$RUNINFO" ]; then
  echo "ERROR: missing $RUNINFO (run Step 2 first)"
  exit 2
fi

echo
python3 - <<'PY'
import csv
import os
from pathlib import Path

acc = os.environ['ACC']
runinfo = Path('zebrafish') / 'metadata' / acc / 'runinfo.csv'
with runinfo.open(newline='', encoding='utf-8') as f:
    r = csv.DictReader(f)
    row = next(r)

print('runinfo columns:', len(row))
print('example Run:', row.get('Run'))
print('example download_path:', row.get('download_path'))
PY
EOF


RIT information technology resources are for the use of the RIT community only. 
By using RIT information technology resources you acknowledge that you have read 
and comply with RIT's Code of Conduct for Computer and Network Use and RIT's 
Information Security Policy and Standards. Use of RIT information technology 
resources may be monitored and unauthorized use is strictly prohibited.


curl:    /usr/bin/curl
python3: /usr/bin/python3

runinfo columns: 47
example Run: SRR34002439
example download_path: https://sra-downloadb.be-md.ncbi.nlm.nih.gov/sos10/sra-pub-zq-1002/SRR034/34002/SRR34002439/SRR34002439.lite.1


In [64]:
%%bash
set -euo pipefail

# Download run files via NCBI `download_path` (from runinfo.csv). No SRA Toolkit required.

SSH="ssh -X pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

ACC="PRJNA1277581"
MAX_SPOTS="10000"  # kept for directory naming consistency; run files are full runs
FORCE_REDOWNLOAD_TEST=0  # set to 1 to overwrite existing downloads

$SSH "REMOTE_REPO=$REMOTE_REPO ACC=$ACC MAX_SPOTS=$MAX_SPOTS FORCE_REDOWNLOAD_TEST=$FORCE_REDOWNLOAD_TEST bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

command -v curl >/dev/null 2>&1 || { echo "ERROR: missing curl"; exit 127; }
PYTHON="/usr/bin/python3"

RUNINFO="zebrafish/metadata/$ACC/runinfo.csv"
RUNS_FILE="zebrafish/metadata/$ACC/runs.test5.smallest_sizeMB.txt"
BASE_DIR="zebrafish/data/$ACC"

[ -f "$RUNINFO" ] || { echo "ERROR: missing $RUNINFO (run Step 2 first)"; exit 2; }
[ -f "$RUNS_FILE" ] || { echo "ERROR: missing $RUNS_FILE (run Step 3 first)"; exit 2; }

mkdir -p "$BASE_DIR"

echo "Downloading run files listed in: $RUNS_FILE"
echo

"$PYTHON" - <<'PY'
import csv
import os
import subprocess
from pathlib import Path

acc = os.environ['ACC']
max_spots = os.environ['MAX_SPOTS']
force = os.environ.get('FORCE_REDOWNLOAD_TEST','0') == '1'

runinfo_path = Path('zebrafish') / 'metadata' / acc / 'runinfo.csv'
runs_file = Path('zebrafish') / 'metadata' / acc / 'runs.test5.smallest_sizeMB.txt'
base_dir = Path('zebrafish') / 'data' / 'fastq' / 'subsets' / acc / f'spots_{max_spots}'

runs = [l.strip() for l in runs_file.read_text(encoding='utf-8').splitlines() if l.strip()]
if not runs:
    raise SystemExit(f'No runs found in {runs_file}')

url_by_run: dict[str, str] = {}
with runinfo_path.open(newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        run = (row.get('Run') or '').strip()
        url = (row.get('download_path') or '').strip()
        if run and url:
            url_by_run[run] = url

missing = [r for r in runs if r not in url_by_run]
if missing:
    raise SystemExit(f'Missing download_path for runs: {missing[:10]} (total missing={len(missing)})')

for run in runs:
    out_dir = base_dir / run / 'sra'
    out_dir.mkdir(parents=True, exist_ok=True)

    url = url_by_run[run]
    fname = url.split('/')[-1]
    dest = out_dir / fname

    # Stable symlink name for downstream tools
    sra_link = out_dir / f'{run}.sra'

    if dest.exists() and not force:
        print('skip (exists):', dest)
    else:
        cmd = ['curl','-fL','--retry','3','--retry-delay','2','--continue-at','-','-o', str(dest), url]
        print('$', ' '.join(cmd))
        subprocess.run(cmd, check=True)

    try:
        if sra_link.exists() or sra_link.is_symlink():
            sra_link.unlink()
        sra_link.symlink_to(dest.name)
    except Exception as e:
        print('WARN: could not create symlink', sra_link, '->', dest.name, ':', e)

print('Done downloading run files.')
PY
EOF

RIT information technology resources are for the use of the RIT community only. 
By using RIT information technology resources you acknowledge that you have read 
and comply with RIT's Code of Conduct for Computer and Network Use and RIT's 
Information Security Policy and Standards. Use of RIT information technology 
resources may be monitored and unauthorized use is strictly prohibited.


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:--  0:00:18 --:--:--     00 --:--:--  0:00:19 --:--:--     0curl: (6) Could not resolve host: sra-downloadb.be-md.ncbi.nlm.nih.gov
  0     0    0     0    0     0      0      0 --:--:--  0:00:09 --:--:--     0curl: (6) Could not resolve host: sra-downloadb.be-md.ncbi.nlm.nih.gov
  0     0    0     0    0     0      0      0 --:--:--  0:00:05 --:--:--     0curl: (6) Could not r     0esolve host: sra-downloadb.be-md.ncbi.nlm.nih.gov
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0curl: (6) Could not resolve host: sra-downloadb.be-md.ncbi.nlm.nih.gov
Traceback (most recent call last):
  File "<stdin>", line 47, in <module>
  File "/usr/lib/python3.13/subprocess.py", line 577, in run
    raise CalledProcessError(retcode, process.args,
                   

$ curl -fL --retry 3 --retry-delay 2 --continue-at - -o zebrafish/data/fastq/subsets/PRJNA1277581/spots_10000/SRR34002427/sra/SRR34002427.lite.1 https://sra-downloadb.be-md.ncbi.nlm.nih.gov/sos10/sra-pub-zq-1002/SRR034/34002/SRR34002427/SRR34002427.lite.1


CalledProcessError: Command 'b'set -euo pipefail\n\n# Download run files via NCBI `download_path` (from runinfo.csv). No SRA Toolkit required.\n\nSSH="ssh -X pzg8794@sequoia.rit.edu"\nREMOTE_REPO="/home/zebrafish"\n\nACC="PRJNA1277581"\nMAX_SPOTS="10000"  # kept for directory naming consistency; run files are full runs\nFORCE_REDOWNLOAD_TEST=0  # set to 1 to overwrite existing downloads\n\n$SSH "REMOTE_REPO=$REMOTE_REPO ACC=$ACC MAX_SPOTS=$MAX_SPOTS FORCE_REDOWNLOAD_TEST=$FORCE_REDOWNLOAD_TEST bash -s" <<\'EOF\'\nset -euo pipefail\ncd "$REMOTE_REPO"\n\ncommand -v curl >/dev/null 2>&1 || { echo "ERROR: missing curl"; exit 127; }\nPYTHON="/usr/bin/python3"\n\nRUNINFO="zebrafish/metadata/$ACC/runinfo.csv"\nRUNS_FILE="zebrafish/metadata/$ACC/runs.test5.smallest_sizeMB.txt"\nBASE_DIR="zebrafish/data/$ACC"\n\n[ -f "$RUNINFO" ] || { echo "ERROR: missing $RUNINFO (run Step 2 first)"; exit 2; }\n[ -f "$RUNS_FILE" ] || { echo "ERROR: missing $RUNS_FILE (run Step 3 first)"; exit 2; }\n\nmkdir -p 

## 4a) Install SRA Toolkit (one-time; server + optional local)

Installs the official prebuilt SRA Toolkit into `zebrafish/tools/sratoolkit/` on the server so `fastq-dump` works for subset extraction.
Run this once per server environment (re-running is safe).


In [65]:
%%bash
set -euo pipefail

# Simple, server-first SRA Toolkit install.
# Installs official prebuilt SRA Toolkit under: /home/zebrafish/zebrafish/tools/sratoolkit

SSH="ssh -X pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

$SSH "REMOTE_REPO=$REMOTE_REPO bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

# If git complains about 'dubious ownership', fix once:
#   git config --global --add safe.directory /home/zebrafish
if ! git status -sb >/dev/null 2>&1; then
  echo "ERROR: git status failed in $REMOTE_REPO"
  echo "If you see 'detected dubious ownership', run:"
  echo "  git config --global --add safe.directory /home/zebrafish"
  git status -sb || true
  exit 2
fi

# Keep repo up to date (non-fatal if it can't fast-forward)
git pull --ff-only || true

TOOLS="$PWD/zebrafish/tools"
mkdir -p "$TOOLS"

LINK="$TOOLS/sratoolkit"
if [ -x "$LINK/bin/fasterq-dump" ] || [ -x "$LINK/bin/fastq-dump" ]; then
  echo "OK: SRA Toolkit already installed at $LINK"
else
  URL="https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/current/sratoolkit.current-ubuntu64.tar.gz"
  TGZ="$TOOLS/sratoolkit.current-ubuntu64.tar.gz"

  echo "Downloading: $URL"
  command -v curl >/dev/null 2>&1 || { echo "ERROR: curl not found"; exit 127; }
  [ -f "$TGZ" ] || curl -fL --retry 3 --retry-delay 2 -o "$TGZ" "$URL"

  echo "Extracting..."
  tar -xzf "$TGZ" -C "$TOOLS"
  FOUND="$(find "$TOOLS" -maxdepth 1 -type d -name 'sratoolkit.*' | sort | tail -n 1)"
  [ -n "$FOUND" ] || { echo "ERROR: extract failed (no sratoolkit.* dir)"; exit 2; }
  ln -sfn "$FOUND" "$LINK"
  echo "Installed: $LINK -> $FOUND"
fi

export PATH="$LINK/bin:$PATH"
echo
echo "fasterq-dump: $(command -v fasterq-dump 2>/dev/null || echo missing)"
echo "fastq-dump:   $(command -v fastq-dump 2>/dev/null || echo missing)"

fasterq-dump --version 2>/dev/null || true
fastq-dump --version 2>/dev/null || true
EOF

# Local install (optional): download the mac tarball to your laptop if you want.
# (Not required for the server workflow.)

RIT information technology resources are for the use of the RIT community only. 
By using RIT information technology resources you acknowledge that you have read 
and comply with RIT's Code of Conduct for Computer and Network Use and RIT's 
Information Security Policy and Standards. Use of RIT information technology 
resources may be monitored and unauthorized use is strictly prohibited.


Already up to date.
OK: SRA Toolkit already installed at /home/zebrafish/zebrafish/tools/sratoolkit

fasterq-dump: /home/zebrafish/zebrafish/tools/sratoolkit/bin/fasterq-dump
fastq-dump:   /home/zebrafish/zebrafish/tools/sratoolkit/bin/fastq-dump
fasterq-dump : 3.3.0

fastq-dump : 3.3.0



## 4b) Optional: Use SRA Toolkit to extract a FASTQ subset (and compare)

If SRA Toolkit is available on `sequoia`, this extracts the first `MAX_SPOTS` spots from the downloaded `*.sra` file (Step 4) and also extracts the same subset by accession. It then hashes the first `MAX_SPOTS` FASTQ records and reports whether they match.


In [ ]:
%%bash
set -euo pipefail

# Extract FASTQ subset with SRA Toolkit and compare
# Note: `fastq-dump` supports `-N/-X` spot ranges; this SRA Toolkit build's `fasterq-dump` does not.

SSH="ssh -X pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

ACC="PRJNA1277581"
MAX_SPOTS="10000"  # number of spots to extract (spot IDs 1..MAX_SPOTS)
FORCE_REEXTRACT=0  # set to 1 to overwrite existing FASTQs

$SSH "REMOTE_REPO=$REMOTE_REPO ACC=$ACC MAX_SPOTS=$MAX_SPOTS FORCE_REEXTRACT=$FORCE_REEXTRACT bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

LINK="$PWD/zebrafish/tools/sratoolkit"
export PATH="$LINK/bin:$HOME/bin:$PATH"

if ! command -v fastq-dump >/dev/null 2>&1; then
  echo "ERROR: fastq-dump not found. Run Step 4a first (install SRA Toolkit)."
  exit 2
fi

echo "fastq-dump: $(command -v fastq-dump)"
fastq-dump --version 2>/dev/null || true

echo

RUNS_FILE="zebrafish/metadata/$ACC/runs.test5.smallest_sizeMB.txt"
SRR="$(head -n 1 "$RUNS_FILE" | tr -d $'\r' | xargs)"
[ -n "$SRR" ] || { echo "ERROR: empty runs list ($RUNS_FILE)"; exit 2; }

echo "SRR=$SRR"

BASE_TEST="zebrafish/data/$ACC/$SRR/subset_spots_${MAX_SPOTS}"
SRA_DIR="$BASE_TEST/sra"
SRC_SRA="$SRA_DIR/${SRR}.sra"

mkdir -p "$SRA_DIR"

# If the .sra file isn't present yet, fetch it now (kept local to this subset folder).
# This avoids requiring a separate "Step 4" pre-download and avoids creating stray SRR* folders in /home/zebrafish.
HAVE_FILE_MODE="1"

# If the .sra file isn't present yet, try to fetch it now (kept local to this subset folder).
# If NCBI endpoints are unreachable from the server, we fall back to accession-only extraction.
if [ ! -f "$SRC_SRA" ]; then
  command -v prefetch >/dev/null 2>&1 || { echo "ERROR: missing prefetch (SRA Toolkit)"; exit 127; }
  echo "Prefetching runfile -> $SRA_DIR"
  # Prefer HTTPS-only to avoid blocked/unstable protocols.
  if ! NCBI_VDB_REMOTE_PROTOCOLS=https prefetch -O "$SRA_DIR" "$SRR"; then
    echo "WARN: prefetch failed (network / NCBI endpoint)."
    echo "WARN: Skipping file-based subset extraction; proceeding with accession-only extraction."
    HAVE_FILE_MODE="0"
  else
    # prefetch usually creates $SRA_DIR/$SRR/<file>; find the first file and link it as ${SRR}.sra
    FOUND=""
    if [ -d "$SRA_DIR/$SRR" ]; then
      FOUND="$(find "$SRA_DIR/$SRR" -maxdepth 1 -type f 2>/dev/null | head -n 1 || true)"
    else
      FOUND="$(find "$SRA_DIR" -maxdepth 1 -type f -name "${SRR}*" 2>/dev/null | head -n 1 || true)"
    fi

    if [ -z "$FOUND" ]; then
      echo "WARN: prefetch finished but no runfile found under $SRA_DIR"
      HAVE_FILE_MODE="0"
    else
      ln -sf "$FOUND" "$SRC_SRA"
    fi
  fi
fi

if [ "$HAVE_FILE_MODE" = "1" ] && [ ! -f "$SRC_SRA" ]; then
  echo "WARN: still missing $SRC_SRA; skipping file-based subset extraction."
  HAVE_FILE_MODE="0"
fi

A_DIR="$BASE_TEST/fastq_subset_from_file"
B_DIR="$BASE_TEST/fastq_subset_from_accession"
mkdir -p "$A_DIR" "$B_DIR"

extract_subset_fastq_dump () {
  local mode="$1"  # file|accession
  local target_dir="$2"
  local input="$3"

  local r1_gz="$target_dir/${SRR}_1.fastq.gz"
  local r2_gz="$target_dir/${SRR}_2.fastq.gz"

  if [ -f "$r1_gz" ] && [ -f "$r2_gz" ] && [ "$FORCE_REEXTRACT" != "1" ]; then
    echo "skip extract ($mode; exists): $target_dir"
    return 0
  fi

  rm -f "$r1_gz" "$r2_gz" || true

  echo "extract ($mode) via fastq-dump -> $target_dir"
  fastq-dump --split-files --gzip -N 1 -X "$MAX_SPOTS" -O "$target_dir" "$input"
}

if [ "$HAVE_FILE_MODE" = "1" ]; then
  extract_subset_fastq_dump file "$A_DIR" "$SRC_SRA"
else
  echo "skip extract (file): no local .sra available"
fi
extract_subset_fastq_dump accession "$B_DIR" "$SRR"

echo
echo "Compare first $MAX_SPOTS records (R1/R2):"

export SRR MAX_SPOTS A_DIR B_DIR HAVE_FILE_MODE
python3 - <<'PY'
import gzip
import hashlib
import os
from pathlib import Path

srr = os.environ["SRR"]
max_records = int(os.environ["MAX_SPOTS"])
a_dir = Path(os.environ["A_DIR"])
b_dir = Path(os.environ["B_DIR"])

have_file_mode = os.environ.get("HAVE_FILE_MODE", "1") == "1"
if not have_file_mode:
    print("SKIP compare: file-mode not available (prefetch failed).")
    print("Accession-only subset FASTQs are in:", b_dir)
    for mate in (1, 2):
        b = b_dir / f"{srr}_{mate}.fastq.gz"
        if not b.exists():
            raise SystemExit(f"Missing expected accession FASTQ: {b}")
        b_md5, b_n, b_hdr = md5_first_records(b, max_records)
        print(f"R{mate}: accession={b.name} records_hashed={b_n} first_header={b_hdr}")
        print(f"  md5(first {max_records} records)={b_md5}")
    raise SystemExit(0)

def md5_first_records(path: Path, n_records: int):
    h = hashlib.md5()
    lines_read = 0
    first_header = ""
    with gzip.open(path, "rb") as f:
        while lines_read < 4 * n_records:
            line = f.readline()
            if not line:
                break
            if lines_read == 0:
                first_header = line.decode("utf-8", "replace").strip()
            h.update(line)
            lines_read += 1
    return h.hexdigest(), lines_read // 4, first_header

for mate in (1, 2):
    a = a_dir / f"{srr}_{mate}.fastq.gz"
    b = b_dir / f"{srr}_{mate}.fastq.gz"
    if not a.exists() or not b.exists():
        raise SystemExit(f"Missing expected FASTQs: {a} or {b}")

    a_md5, a_n, a_hdr = md5_first_records(a, max_records)
    b_md5, b_n, b_hdr = md5_first_records(b, max_records)

    print(f"R{mate}:")
    print(f"  file:      {a.name}  records_hashed={a_n}  first_header={a_hdr}")
    print(f"  accession: {b.name}  records_hashed={b_n}  first_header={b_hdr}")
    print(f"  md5(first {max_records} records):")
    print(f"    file={a_md5}")
    print(f"    acc ={b_md5}")
    print(f"    MATCH={a_md5 == b_md5}")
PY
EOF


RIT information technology resources are for the use of the RIT community only. 
By using RIT information technology resources you acknowledge that you have read 
and comply with RIT's Code of Conduct for Computer and Network Use and RIT's 
Information Security Policy and Standards. Use of RIT information technology 
resources may be monitored and unauthorized use is strictly prohibited.


fastq-dump: /home/zebrafish/zebrafish/tools/sratoolkit/bin/fastq-dump
fastq-dump : 3.3.0


SRR=SRR34002427
Prefetching runfile -> zebrafish/data/PRJNA1277581/SRR34002427/subset_spots_10000/sra
2026-02-19T04:46:26 prefetch.3.3.0: 1) Resolving 'SRR34002427'...
2026-02-19T04:46:27 prefetch.3.3.0: Current preference is set to retrieve SRA Normalized Format files with full base quality scores


2026-02-19T04:56:57 prefetch.3.3.0 int: connection not found while validating within network system module - cannot open remote file [https://sra-download-internal.ncbi.nlm.nih.gov/sos10/sra-pub-zq-1002/SRR034/34002/SRR34002427/SRR34002427.lite.1]


## 5) Validate the outputs (quick checks)

We verify the downloaded run file exists for each SRR. If you also ran Step 4b, we verify the extracted `*.fastq.gz` subset files are readable and show the first FASTQ record.


In [ ]:
%%bash
set -euo pipefail
# Validate outputs: confirm the .sra exists and the extracted FASTQ subset gzip files are readable.
# Prints the first FASTQ record as a quick sanity check.


SSH="ssh -X pzg8794@sequoia.rit.edu"
REMOTE_REPO="/home/zebrafish"

ACC="PRJNA1277581"
MAX_SPOTS="10000"

$SSH "REMOTE_REPO=$REMOTE_REPO ACC=$ACC MAX_SPOTS=$MAX_SPOTS bash -s" <<'EOF'
set -euo pipefail
cd "$REMOTE_REPO"

RUNS_FILE="zebrafish/metadata/$ACC/runs.test5.smallest_sizeMB.txt"
BASE_DIR="zebrafish/data/$ACC"

[ -f "$RUNS_FILE" ] || { echo "ERROR: missing $RUNS_FILE"; exit 2; }

while read -r SRR; do
  [ -n "$SRR" ] || continue
  d="$BASE_DIR/$SRR"
  echo
  echo "== $SRR =="
  [ -d "$d" ] || { echo "Missing dir: $d"; exit 2; }

  echo "SRA download(s):"
  ls -lh "$d/sra" || { echo "Missing: $d/sra (run Step 4)"; exit 2; }
  [ -f "$d/sra/${SRR}.sra" ] || { echo "Missing: $d/sra/${SRR}.sra"; exit 2; }
  ls -lh "$d/sra/${SRR}.sra"

  # If Step 4b was run, validate FASTQs too
  for sub in "$d/fastq_subset_from_file"; do
    if [ -d "$sub" ]; then
      echo
      echo "FASTQ subset in: $sub"
      ls -lh "$sub" | head
      for f in "$sub"/*.fastq.gz; do
        [ -f "$f" ] || continue
        echo
        echo "gzip -t $f"
        gzip -t "$f"
        echo "first record:"
        # Avoid failing under `set -o pipefail` due to SIGPIPE when `head` exits early
        set +o pipefail
        zcat "$f" | head -n 4
        set -o pipefail
      done
    fi
  done

done < "$RUNS_FILE"

echo
echo "Validation complete."
EOF


## Next steps

- If this test looks good, run full downloads on the server (prefer `prefetch` + `fasterq-dump`) and keep the raw data in `data/` (gitignored).
- Keep using `metadata/PRJNA1277581/runinfo.filtered.csv` as the source of truth for which runs are in-scope.

### Cleanup (demo only)

This notebook is an *exploration* / comparison. If you ran the demo steps above, you can remove the subset outputs to keep the shared `zebrafish/data/` folder clean.


In [ ]:
%%bash
set -euo pipefail

# Delete subset demo outputs created by this notebook (safe if missing).
ACC="PRJNA1277581"
MAX_SPOTS="10000"

# If you downloaded multiple SRRs, this removes only the subset subfolders, not the full FASTQs.
find "zebrafish/data/$ACC" -maxdepth 2 -type d -name "subset_spots_${MAX_SPOTS}" -print -exec rm -rf {} + 2>/dev/null || true
echo "Cleanup done."
